traitement données base gravity répliquée de WR avec base gravity du CEPII pour données économiques (gdp) 

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# chemins 
path_r_output = "/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/gravity_data_WR_replique.csv" # Ton fichier 229k lignes
path_gravity  = "/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv" # La base CEPII brute

# chargement et dictionnaires pour simplifier 
print("Chargement des données...")
df_main = pd.read_csv(path_r_output)
df_grav = pd.read_csv(path_gravity, low_memory=False)

# On prépare le "Dictionnaire Économique" (Source CEPII)
# On ne garde que les colonnes utiles et on dédoublonne (Crucial !)
cols_eco = ['year', 'iso3_o', 'iso3_d', 'gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d']
df_eco = df_grav[cols_eco].drop_duplicates(subset=['year', 'iso3_o', 'iso3_d'])

print(f"Dictionnaire Éco prêt : {len(df_eco)} lignes uniques.")

#   FUSION DES VALEURS COURANTES (t) 
# On fusionne sur Année + ISO Origine + ISO Destination
# Hypothèse : dans ton fichier R, les colonnes s'appellent 'year0', 'orig', 'dest'
# Si elles s'appellent autrement (ex: 'year'), change 'left_on' ci-dessous.

df_final = pd.merge(
    df_main, 
    df_eco, 
    left_on=['year0', 'orig', 'dest'],   # Clés du fichier R
    right_on=['year', 'iso3_o', 'iso3_d'], # Clés du fichier Gravity
    how='left'
)

# Nettoyage des colonnes doublons après fusion
df_final = df_final.drop(columns=['year', 'iso3_o', 'iso3_d']) 

# 4. FUSION DES LAGS (t-5)
# Astuce : Pour avoir le PIB de 2000 sur la ligne 2005, on prend les données 2000
# et on dit qu'elles "appartiennent" à l'année cible 2005.

df_lag = df_eco.copy()
df_lag['year_target'] = df_lag['year'] + 5 # On décale l'année de +5 ans

# On renomme les variables pour qu'elles aient le suffixe "_lag"
cols_lag = {
    'gdp_o': 'gdp_o_lag', 'gdp_d': 'gdp_d_lag',
    'gdpcap_o': 'gdpcap_o_lag', 'gdpcap_d': 'gdpcap_d_lag'
}
df_lag = df_lag.rename(columns=cols_lag)

# Fusion sur l'année DÉCALÉE
df_final = pd.merge(
    df_final,
    df_lag[['year_target', 'iso3_o', 'iso3_d'] + list(cols_lag.values())],
    left_on=['year0', 'orig', 'dest'],
    right_on=['year_target', 'iso3_o', 'iso3_d'],
    how='left'
)

df_final = df_final.drop(columns=['year_target', 'iso3_o', 'iso3_d'])

#  CALCUL DES LOGS  
vars_to_log = ['gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 
               'gdp_o_lag', 'gdp_d_lag', 'gdpcap_o_lag', 'gdpcap_d_lag']

for col in vars_to_log:
    # Log sécurisé (met NaN si valeur <= 0 ou manquante)
    df_final[f'log_{col}'] = np.log(df_final[col].mask(df_final[col] <= 0))

# EXPORT 
print("-" * 30)
print(f"Dimensions finales : {df_final.shape}")
print(f"PIB manquants (courant) : {df_final['gdp_o'].isna().sum()} / {len(df_final)}")
print(f"PIB manquants (lag t-5) : {df_final['gdp_o_lag'].isna().sum()} (Normal pour 1990)")

df_main = pd.read_csv("/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/FINAL_GRAVITY_WITH_GDP.csv")
#print(gravity_data.head())


df_final= df_final.drop(columns=['iso3.x','iso3.y'])


df_final.rename(columns={'cod_o': 'iso_o', 'cod_d': 'iso_d','year0':'year'}, inplace=True)


df_final.to_csv("/Users/romain/Desktop/Projets DS/ProjetStat/data/data_final/FINAL_GRAVITY_WITH_GDP.csv", index=False)
print("✅ Fichier sauvegardé.")

possible chunks si trop long à mettre en cellule de code: 

import pandas as pd
import polars as pl

#df_gravity= pd.read_csv('/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv',)
#df_gravity=df_gravity[df_gravity['year']==2010 | df_gravity['year']==2015 | df_gravity['year']==2005 | df_gravity['year']==2000 | df_gravity['year']==1995 | df_gravity['year']==1990]


#CHUNKS FAIT PAR GEMINI

PATH_GRAVITY = '/Users/romain/Desktop/Projets DS/ProjetStat/data/GravityCEPII/Gravity_V202211.csv'
ANNEES_CIBLES = [1990, 1995, 2000, 2005, 2010, 2015]

# Liste pour stocker les petits morceaux filtrés
chunks_list = []



# chunksize=100000 signifie qu'on lit 100 000 lignes à la fois
# low_memory=False permet d'éviter les warnings sur les types mixtes (ex: 6.5 vs 6)
with pd.read_csv(PATH_GRAVITY, chunksize=100000, low_memory=False) as reader:
    for i, chunk in enumerate(reader):
        # 1. On ne garde que les années cibles dans ce morceau
        chunk_filtered = chunk[chunk['year'].isin(ANNEES_CIBLES)]
        
        # 2. On ajoute le résultat à notre liste (si le morceau n'est pas vide)
        if not chunk_filtered.empty:
            chunks_list.append(chunk_filtered)
            
        # Petit indicateur de progression (optionnel)
        if i % 10 == 0:
            print(f"   ... Traitement du bloc {i}")


if len(chunks_list) > 0:
    df_gravity = pd.concat(chunks_list, ignore_index=True)
    print(f"Terminé ! Taille finale : {df_gravity.shape}")
    print("Années présentes :", sorted(df_gravity['year'].unique()))
else:
    print("Attention : Aucune donnée trouvée pour ces années.")

# Affichage des premières lignes
print(df_gravity.head())